In [ ]:
pip install nltk youtube-transcript-api transformers torch

In [ ]:
import re
import heapq
import nltk
import torch
from collections import defaultdict
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, T5Tokenizer, T5ForConditionalGeneration

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
def extract_video_id(youtube_url):
    match = re.search(r"(?:v=|\/)([0-9A-Za-z_-]{11}).*", youtube_url)
    return match.group(1) if match else None

# Fetch transcript from YouTube video
def get_youtube_transcript(youtube_url):
    video_id = extract_video_id(youtube_url)
    if not video_id:
        return "Invalid YouTube URL"
    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([t['text'] for t in transcript])
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def textrank_summarize(text, summary_length=1):
    sentences = sent_tokenize(text)
    stop_words = set(stopwords.words("english"))
    word_frequencies = defaultdict(int)

    for sentence in sentences:
        for word in word_tokenize(sentence.lower()):
            if word not in stop_words and word.isalnum():
                word_frequencies[word] += 1

    max_freq = max(word_frequencies.values(), default=1)
    for word in word_frequencies:
        word_frequencies[word] /= max_freq

    sentence_scores = defaultdict(int)
    for sentence in sentences:
        for word in word_tokenize(sentence.lower()):
            if word in word_frequencies:
                sentence_scores[sentence] += word_frequencies[word]

    summary_sentences = heapq.nlargest(summary_length, sentence_scores, key=sentence_scores.get)
    return " ".join(summary_sentences)


In [ ]:
def summarize_text(tokenizer, model, text):
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt")
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, early_stopping=True)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
pegasus_checkpoint = "google/pegasus-large"
bart_checkpoint = "sshleifer/distilbart-cnn-12-6"
pegasus_tokenizer = AutoTokenizer.from_pretrained(pegasus_checkpoint)
pegasus_model = AutoModelForSeq2SeqLM.from_pretrained(pegasus_checkpoint)
bart_tokenizer = AutoTokenizer.from_pretrained(bart_checkpoint)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(bart_checkpoint)

In [ ]:
def t5_summarize(text, max_length=100):
    tokenizer = T5Tokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small").to("cuda" if torch.cuda.is_available() else "cpu")

    chunks = chunk_text(text, max_tokens=450)
    summaries = []

    for chunk in chunks:
        input_text = "summarize: " + chunk
        input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, legacy= False).to(model.device)
        summary_ids = model.generate(input_ids, max_length=max_length, num_beams=4, early_stopping=True)
        summaries.append(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

    return " ".join(summaries)

In [ ]:
youtube_url = "https://www.youtube.com/watch?v=_T73DugW_D8"
transcript = get_youtube_transcript(youtube_url)

if not transcript.startswith("Error"):
    print("\n🔹 NLTK TextRank Summary:")
    print(textrank_summarize(transcript))

    print("\n🔹 Pegasus Summary:")
    print(summarize_text(pegasus_tokenizer, pegasus_model, transcript))

    print("\n🔹 BART Summary:")
    print(summarize_text(bart_tokenizer, bart_model, transcript))

    print("\n🔹 T5 Summary:")
    print(t5_summarize(transcript))
else:
    print(transcript)
